# 10 — Indicators Module Quickstart

Quick tour of every indicator group in the Swing Screener framework. See the [indicators README](https://github.com/matteolongo/swing_screener/blob/main/src/swing_screener/indicators/README.md) for full documentation.

In [ ]:
from __future__ import annotations
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

from swing_screener.data.providers import get_market_data_provider
from swing_screener.indicators.trend import TrendConfig, compute_trend_features
from swing_screener.indicators.momentum import MomentumConfig, compute_momentum_features
from swing_screener.indicators.volatility import VolatilityConfig, compute_volatility_features
from swing_screener.indicators.swings import swing_points
from swing_screener.indicators.volume_pressure import intrabar_pressure, trailing_volume_ratio
from swing_screener.indicators.vwap import anchored_vwap
from swing_screener.indicators.candles import detect_patterns
from swing_screener.indicators.setup_quality import compute_setup_quality
from swing_screener.indicators.exhaustion import compute_exhaustion_score

This notebook loads OHLCV for SPY and AAPL, then runs every indicator family.

**Dependencies:** `swing_screener[dev]` installed via `uv sync`; yfinance for data fetching.

In [ ]:
provider = get_market_data_provider()
ohlcv = provider.fetch_ohlcv(["SPY", "AAPL"], "2023-01-01", "2024-12-31")
ohlcv.head()

### Trend Indicators

SMA-based trend detection: SMA20/50/200, trend_ok flag, distance from SMAs.

In [ ]:
trend_df = compute_trend_features(ohlcv, TrendConfig())
trend_df

### Momentum + Relative Strength

6-month and 12-month price momentum, plus 6-month relative strength vs SPY.

In [ ]:
mom_df = compute_momentum_features(ohlcv, MomentumConfig())
mom_df

### Volatility / ATR

ATR14 and ATR% using Wilder's smoothing per ticker.

In [ ]:
vol_df = compute_volatility_features(ohlcv, VolatilityConfig())
vol_df

### Swing Detection

Fractal swing-pivot detection: finds the most recent confirmed pivot high and low per ticker.

In [ ]:
for ticker in ["SPY", "AAPL"]:
    sp = swing_points(ohlcv["High"][ticker], ohlcv["Low"][ticker])
    print(f"{ticker}: swing_high={sp.swing_high}, swing_low={sp.swing_low}")

### Volume Pressure

Intrabar close-location (buy/sell proxy) and trailing volume ratio for the latest bar.

In [ ]:
for ticker in ["SPY", "AAPL"]:
    h = float(ohlcv["High"][ticker].iloc[-1])
    l = float(ohlcv["Low"][ticker].iloc[-1])
    c = float(ohlcv["Close"][ticker].iloc[-1])
    pressure = intrabar_pressure(h, l, c)
    vol = ohlcv["Volume"][ticker].values
    idx = len(vol) - 1
    ratio = trailing_volume_ratio(vol, idx)
    print(f"{ticker}: intrabar_pressure={pressure:.3f}, trailing_volume_ratio={ratio}")

### VWAP

Volume-weighted average price anchored to the full available history.

In [ ]:
for ticker in ["SPY", "AAPL"]:
    vwap = anchored_vwap(
        ohlcv["High"][ticker], ohlcv["Low"][ticker],
        ohlcv["Close"][ticker], ohlcv["Volume"][ticker]
    )
    last_close = float(ohlcv["Close"][ticker].iloc[-1])
    print(f"{ticker}: anchored_vwap={vwap:.2f}, last_close={last_close:.2f}, diff_pct={(last_close/vwap - 1)*100:.2f}%")

### Candle Patterns

Deterministic candlestick pattern detection with volume-pressure confirmation.

In [ ]:
patterns = detect_patterns(ohlcv, tickers=["SPY", "AAPL"])
for ticker, pat_list in patterns.items():
    print(f"\n{ticker} ({len(pat_list)} patterns):")
    for p in pat_list[-5:]:
        print(f"  {p.date} | {p.name:20s} | {p.direction:8s} | key_level={p.key_level:.2f} | context={p.context} | vol_confirmed={p.volume_confirmed}")

### Setup Quality + Exhaustion

Setup quality metrics (consolidation tightness, close location, breakout extension) and composite trend exhaustion score.

In [ ]:
setup_df = compute_setup_quality(ohlcv)
setup_df

In [ ]:
for ticker in ["SPY", "AAPL"]:
    result = compute_exhaustion_score(
        ohlcv["Close"][ticker], ohlcv["High"][ticker],
        ohlcv["Low"][ticker], ohlcv["Volume"][ticker]
    )
    print(f"{ticker}: score={result.score}, label={result.label}")
    print(f"  components: {result.components}")

### Summary

All feature columns joined into a single overview table.

In [ ]:
summary = trend_df.join(mom_df, how="outer").join(vol_df, how="outer")
summary = summary.join(setup_df, how="outer")
summary